In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import re
import pandas as pd
#from datasets import load_dataset
import time
import pickle
from tqdm import tqdm

## GPT HELP

In [3]:
df1 = pd.read_pickle("/content/drive/MyDrive/NLP_project/Abhishek (Mutli tool Chained- hotpot)/First_QAWC.pkl")
df1

,Q,A,W,C
0,"The Dutch-Belgian television series that ""Hous...",6,House of Anubis > House of Anubis is a mystery...,First search for [Het Huis Anubis first aired ...
1,What would be the length of the track where th...,10.213,2013 Liqui Moly Bathurst 12 Hour > The 2013 Li...,First search [Mount Panorama Circuit -Wiki-> y...
2,"The 1988 American comedy film, The Great Outdo...",4,The Great Outdoors (film) > The Great Outdoors...,First Search [Annette Bening -Wiki-> y1]. She ...
3,"Dua Lipa, an English singer, songwriter and mo...",17,Dua Lipa > Her self-titled debut studio album...,First Search [ Dua Lipa New Rules release -Wik...
4,How old would be the female main protagonist o...,22,Catching Fire > Catching Fire is a 2009 scienc...,First Search for [Catching Fire Suzanne Collin...
...,...,...,...,...
13907,A Grupė is the highest level of women's volley...,30000,A Grupė (Volleyball) > The A Grupė is highest ...,First search for [A Grupe -Wiki-> y1]. Now sea...
13908,The 1968 Baltimore Orioles season included a r...,35,1968 Baltimore Orioles season > The team was ...,First search for [1968 Baltimore Orioles seaso...
13909,How many stories are in the hotel that hosted ...,600,10th IIFA Awards > The 10th International Indi...,First search for [10th IIFA Awards -Wiki-> y1]...
13931,Multiply the number of years that Cocteau Twin...,10008,Cocteau Twins > Cocteau Twins were a Scottish ...,First Search [Cocteau Twins -Wiki-> y1]. Band ...


In [4]:
first100 = df1.iloc[:500]
first100

,Q,A,W,C
0,"The Dutch-Belgian television series that ""Hous...",6,House of Anubis > House of Anubis is a mystery...,First search for [Het Huis Anubis first aired ...
1,What would be the length of the track where th...,10.213,2013 Liqui Moly Bathurst 12 Hour > The 2013 Li...,First search [Mount Panorama Circuit -Wiki-> y...
2,"The 1988 American comedy film, The Great Outdo...",4,The Great Outdoors (film) > The Great Outdoors...,First Search [Annette Bening -Wiki-> y1]. She ...
3,"Dua Lipa, an English singer, songwriter and mo...",17,Dua Lipa > Her self-titled debut studio album...,First Search [ Dua Lipa New Rules release -Wik...
4,How old would be the female main protagonist o...,22,Catching Fire > Catching Fire is a 2009 scienc...,First Search for [Catching Fire Suzanne Collin...
...,...,...,...,...
755,What would be the population of the city where...,103297,Lansing State Journal > The Lansing State Jour...,First search for [ Lansing State Journal -Wiki...
756,What year would the debut single of the singer...,2014,"3AM (Pull Up) > ""3am (Pull Up)"" is a song reco...",First search for [Charli XCX -Wiki-> y1]. Sear...
757,Between what years would Charles Hutchison and...,1925,Charles Hutchison > He appeared in 49 films b...,First search for [Charles Hutchison -Wiki-> y1...
758,How old would the American social entrepeneur ...,54,"Rachel Jacobs > Rachel Jacobs (October 3, 1975...",Search for [Rachel Jacobs -Wiki-> y1]. Search ...


In [5]:
prompt_examples="""1.
Q: The Dutch-Belgian television series that "House of Anubis" was based on first aired in how many years after 2000?
A: 6
W: House of Anubis > House of Anubis is a mystery television series developed for Nickelodeon based on the Dutch-Belgian television series "Het Huis Anubis". | Het Huis Anubis >  It first aired in September 2006 and the last episode was broadcast on December 4, 2009.
C: First search for [Het Huis Anubis first aired -Wiki-> y1]. First episode released in [y1 -QA(Which year was the fist episode aired ?)-> y2]. Number of years it aired after 2000 [y2 - 2000 = y3]. The answer is y3.

2.
Q: What would be the length of the track where the 2013 Liqui Moly Bathurst 12 Hour was staged, if it would have been 4km longer?
A: 10.213
W: 2013 Liqui Moly Bathurst 12 Hour > The 2013 Liqui Moly Bathurst 12 Hour was an endurance race for a variety of GT and touring car classes, including: GT3 cars, GT4 cars, Group 3E Series Production Cars and Dubai 24 Hour cars. | 2013 Liqui Moly Bathurst 12 Hour >  The event, which was staged at the Mount Panorama Circuit, near Bathurst, in New South Wales, Australia on 10 February 2013, was the eleventh running of the Bathurst 12 Hour. | Mount Panorama Circuit > Mount Panorama Circuit is a motor racing track located in Bathurst, New South Wales, Australia. | Mount Panorama Circuit >  The 6.213 km long track is technically a street circuit, and is a public road, with normal speed restrictions, when no racing events are being run, and there are many residences which can only be accessed from the circuit.
C: First search [Mount Panorama Circuit -Wiki-> y1]. Length of circuit is [y1 -QA(what is the length of Mount Panorama Circuit ?)-> y2]. Length after adding 4km will be [4 + y2 = y3]. The answer is y3.

3.
Q: The 1988 American comedy film, The Great Outdoors, starred a four-time Academy Award nominee, who received a star on the Hollywood Walk of Fame in how many years before 2010?
A: 4
W: The Great Outdoors (film) > The Great Outdoors is a 1988 American comedy film directed by Howard Deutch, and written and produced by John Hughes. | The Great Outdoors (film) >  It stars Dan Aykroyd, John Candy, Stephanie Faracy and Annette Bening in her film debut. | Annette Bening > Annette Carol Bening (born May 29, 1958) is an American actress. | Annette Bening >  She is a four-time Academy Award nominee; for "The Grifters" (1990), "American Beauty" (1999), "Being Julia" (2004) and "The Kids Are All Right" (2010). | Annette Bening >  In 2006, she received a star on the Hollywood Walk of Fame.
C: First Search [Annette Bening -Wiki-> y1]. She received star on the Hollywood Walk of Fame on [y1 -QA(When did she recieved star on Hollywood Walk of Fame ?)-> y2]. Number of years before 2010 [2010 - y2 = y3]. The answer is y3.

4.
Q: Dua Lipa, an English singer, songwriter and model, the album spawned the number-one single "New Rules" is a song by English singer Dua Lipa from her eponymous debut studio album, released in how many years after 2000?
A: 17
W: Dua Lipa >  Her self-titled debut studio album was released on 2 June 2017. | New Rules (song) > "New Rules" is a song by English singer Dua Lipa from her eponymous debut studio album (2017).
C: First Search [ Dua Lipa New Rules release -Wiki-> y1]. New rules was released in [y1 -QA(In which year was New Rules song released ?)-> y2]. Number of years from 2000 after its released is [y2 - 2000 = y3]. The answer is y3.

5.
Q: How old would be the female main protagonist of Catching Fire, if she was born 6 years before?
A: 22
W: Catching Fire > Catching Fire is a 2009 science fiction young adult novel by the American novelist Suzanne Collins, the second book in "The Hunger Games trilogy". | The Hunger Games (novel) >  It is written in the voice of 16-year-old Katniss Everdeen, who lives in the future, post-apocalyptic nation of Panem in North America.
C: First Search for [Catching Fire Suzanne Collins -Wiki-> y1]. Search for her age at that time [y1 -QA(What was the age of Katniss Everdeen at that time ?)-> y2]. She would how many years old is she was born 6 years before [y2 + 6 = y3]. The answer is y3.

6.
Q: How many years before 1947 was the creator of the current arrangement of the "Simpson's Theme" born?
A: 6
W: The Simpsons Theme > "The Simpsons" Theme", also referred to as "The Simpsons" Main Title Theme" in album releases, is the theme music of the animated television series "The Simpsons". | The Simpsons Theme >  The theme, as used for the opening sequence, was re-arranged during season 2, and the current arrangement by Alf Clausen was introduced at the beginning of the third season. | Alf Clausen > Alf Heiberg Clausen (born March 28, 1941) is an American film and television composer.
C: First find [Simpson's Theme creator -Wiki-> y1]. Now search for when was he born [y1 -QA(When was Alf Heiberg Clausen born ?)-> y2]. Now find how many years before 1947 he was born [1947 - y2 = y3]. The final answer is y3.

7.
Q: The American Pre-Code comedy film featuring an American actress, dancer, and singer, widely known for performing in films and RKO's musical films, was released in how many years after 1925?
A: 7
W: Hat Check Girl > Hat Check Girl is a 1932 American Pre-Code comedy film directed by Sidney Lanfield and written by Barry Conners and Philip Klein. | Hat Check Girl >  The film stars Sally Eilers, Ben Lyon, Ginger Rogers and Monroe Owsley. | Ginger Rogers > Ginger Rogers (born Virginia Katherine McMath; July 16, 1911 – April 25, 1995) was an American actress, dancer, and singer, widely known for performing in films and RKO's musical films, partnered with Fred Astaire.
C: First Search for [American Pre-Code comedy RKO musical film -Wiki-> y1]. Now search for year it was released [y1 -QA(When was Hat Check Girl Released ?)-> y2]. Now find number of years after 1925 it was released [y2 - 1925 = y3]. The answer is y3.

8.
Q: How many years after 1950 was the winner of the 2016 Marrakesh ePrix born?
A: 38
W: 2016 Marrakesh ePrix >  The 33-lap race was won by e.Dams-Renault driver Sébastien Buemi, who started from the seventh position. | Sébastien Buemi > Sébastien Olivier Buemi (born 31 October 1988) is a Swiss professional racing driver, who formerly competed for Scuderia Toro Rosso in Formula One.
C: First search for [2016 Marrakesh ePrix winner -Wiki-> y1]. Now search for when he was born [y1 -QA(When was Sébastien Buemi born ?)-> y2]. Now search for how many years after 1950 he was born so subtract [y2 - 1950 = y3]. The answer is y3.
"""

In [6]:
prompt_rows = first100[['Q','A','W']].to_dict('records')

In [7]:
gemini_prompt=f"""You are an agent that converts Questions, Answers, and corresponding Wikipedia Knowledge to a Chain of Abstractions. The intuition is to get to the Answer using the titles searched in the  Wikipedia knowledge.

Rules to generate C:
1.  You have 3 tools, Wiki, QA and Mathematical. Use **all tools** to derive C. Tool Explanations are:
	Wiki Tool to get relevant articles from Wikipedia. Format: [search query -Wiki-> search query output]
	QA tool to get the focused answer from the Wikipedia articles. Format:  [input context -QA(question)-> output]
  Mathematical tool solves any mathematical computes over the informtion returned from QA tool Format: [polynomial expression] eg:[y1 + 20 = y2]
2. W is formatted to put the title before '>' and content after '>' and separate articles using '|'.
3. Use the outputs from Wiki tool as input context for QA tool. And use QA tool output as input for mathematical tool The final answer is always output of Mathematical tool. **This is crucial** please try to follow this format
4. You can use page titles but cannot use page content information given in W to form Wiki search Queries.
5. You cannot use answer (A) in the tool queries directly.
6. You can only use first wikipedia title in the tool queries directly and cannot use any subsequent titles directly in tool queries
7. Utilize the Chain of Thought process from the Wikipedia Knowledge given. Learn what titles need to be searched in the Wiki tool and asked in the QA tool to get to the final answer. Conclude by ** The answer is <variable_name>.**

Your task to generate Abstractions(C)  for the given Question (Q) using the Wiki tool, the QA tool and the mathematical tool using the rules.

Examples to generate Chain of Abstraction( C)  given Q, A and W are:
### EXAMPLES
{prompt_examples}

### TASK
Now  generate C for the test examples below. Please Write '###DONE' when you have completed generating C for all of the examples below.
Respond in this format only:
Test Example <example number> C:

"""

def generate_continuation_gemini(prompt_rows):
    prompt = gemini_prompt

    for idx, row in enumerate(prompt_rows):
        prompt += f"Test Example {idx + 1}\n"
        prompt += f"Q: {row['Q']}\n"
        prompt += f"A: {row['A']}\n"
        prompt += f"W: {row['W']}\n"
        prompt += "\n"
    return prompt + "### RESPONSE\n"

##CREATE PROMPT

In [8]:
generated_dataset = [] #INITIALIZATION

In [134]:
selected_rows=prompt_rows[400:410] #CHOOSE ROWS

In [135]:
input_prompt=generate_continuation_gemini(selected_rows)
print(input_prompt)

You are an agent that converts Questions, Answers, and corresponding Wikipedia Knowledge to a Chain of Abstractions. The intuition is to get to the Answer using the titles searched in the  Wikipedia knowledge.

Rules to generate C:
1.  You have 3 tools, Wiki, QA and Mathematical. Use **all tools** to derive C. Tool Explanations are:
	Wiki Tool to get relevant articles from Wikipedia. Format: [search query -Wiki-> search query output]
	QA tool to get the focused answer from the Wikipedia articles. Format:  [input context -QA(question)-> output]
  Mathematical tool solves any mathematical computes over the informtion returned from QA tool Format: [polynomial expression] eg:[y1 + 20 = y2]
2. W is formatted to put the title before '>' and content after '>' and separate articles using '|'.
3. Use the outputs from Wiki tool as input context for QA tool. And use QA tool output as input for mathematical tool The final answer is always output of Mathematical tool. **This is crucial** please try

In [130]:
paste_response="""
Test Example 1 C:
First search for [Euroinvestor -Wiki-> y1]. Search for the year the bank was established [y1 -QA(When was Euroinvestor established ?)-> y2]. Now find the year it would be 58 years old [y2 + 58 = y3]. The answer is y3.

Test Example 2 C:
First search for [Richard Godwin project -Wiki-> y1]. Search for the cost of the project [y1 -QA(What was the cost of the NS Savannah project ?)-> y2]. Now find the cost if it was 60% more expensive [y2 + (0.6 * y2) = y3]. The answer is y3.

Test Example 3 C:
First search for [Concord Naval Weapons Station -Wiki-> y1]. Search for the year it was established [y1 -QA(When was Concord Naval Weapons Station established ?)-> y2]. Now find the year it would be 100 years old [y2 + 100 = y3]. The answer is y3.

Test Example 4 C:
First search for [Princess Maria-Olympia of Greece and Denmark -Wiki-> y1]. Search for the year Constantine II was last king of Greece [y1 -QA(When was Constantine II the last king of Greece ?)-> y2]. Now find the year multiplied by 2 [1973 * 2 = y3]. The answer is y3.

Test Example 5 C:
First search for [Get Rich or Die Tryin' (film) -Wiki-> y1]. Now search for the artist's name [y1 -QA(Who starred in Get Rich or Die Tryin' ?)-> y2]. Search for the number of nominations at the 54th Grammy Awards [y2 -QA(How many Grammy Awards nominations did 50 Cent receive for the soundtrack ?)-> y3]. The answer is y3.

Test Example 6 C:
First search for [Three Hundred Big Boys Futurama -Wiki-> y1]. Now search for the director of the episode [y1 -QA(Who directed Three Hundred Big Boys ?)-> y2]. Now search for the number of segments in the episode [y1 -QA(How many segments are there in Three Hundred Big Boys ?)-> y3]. The answer is y3.

Test Example 7 C:
First search for [Tekno Team 2000 -Wiki-> y1]. Now search for Erik Watts birth year [y1 -QA(When was Erik Watts born ?)-> y2]. Search for the number of championships won [y1 -QA(How many championships did Tekno Team 2000 win ?)-> y3]. The answer is y3.

Test Example 8 C:
First search for [Joseph Smith (explorer) -Wiki-> y1]. Now search for the year York Factory was built [y1 -QA(When was York Factory built ?)-> y2]. Now find the number of locations it had when it was first used [y1 -QA(How many locations did York Factory have when it was first used ?)-> y3]. The answer is y3.

Test Example 9 C:
First search for [Saw 3D -Wiki-> y1]. Now search for the release year of the movie [y1 -QA(When was Saw 3D released ?)-> y2]. Search for the number of releases that week [y1 -QA(How many times was Saw 3D released that week ?)-> y3]. The answer is y3.

Test Example 10 C:
First search for [Donald Cammell -Wiki-> y1]. Now search for Peter Bogdanovich birth year [y1 -QA(When was Peter Bogdanovich born ?)-> y2]. Find the age of Donald Cammell if he were still alive today [current year - 1996 = y3]. Find the age of Peter Bogdanovich if he were still alive today [current year - y2 = y4]. Find the difference in their ages [y3 - y4 = y5]. The answer is y5.

###DONE


"""

In [131]:
paste_response=paste_response.replace("C:\n","C:")
contents = re.findall(r'C:(.*)', paste_response)
if len(contents)==10:
  print('FOUND ALL 8')
  # Add each element of content to the corresponding row
  for i, row in enumerate(selected_rows):
      row['C'] = contents[i].strip()

FOUND ALL 8


In [132]:
generated_dataset.extend(selected_rows)

In [133]:
len(generated_dataset)

200

In [136]:
generated_dataset

[{'Q': 'How many countries that are not Member states of the Commonwealth of Nations utilize English as a sole official language or one of the official languages?',
  'A': '10',
  'W': 'English in the Commonealth of Nations > The use of the English language in most member countries of the Commonwealth of Nations was inherited from British colonisation. | English in the Commonealth of Nations >  English is spoken as a first or second language in most of the Commonwealth. | English in the Commonwealth of Nations >  In a few countries, such as Cyprus and Malaysia, it does not have official status, but is widely used as a lingua franca. | English in the Commonwealth of Nations >  Mozambique is an exception – although English is widely spoken there, it is a former Portuguese colony which joined the Commonwealth in 1996. | List of countries where English is an official language > In addition to the 26 Commonwealth of Nations, there are 10 other countries where English is an official language

In [ ]:
## at the end save generated dataset

In [138]:
df = pd.DataFrame(generated_dataset)
df.to_pickle('/content/drive/MyDrive/NLP_project/abhishek2.pkl')

## MUKTI_TOOL ABHISHEK

In [ ]:
df1 = pd.read_pickle("/content/drive/MyDrive/NLP_project/Abhishek (Mutli tool Chained- hotpot)/First_QAWC.pkl")
df1